# FreshCart ELT Pipeline on Snowflake
Complete pipeline: Extract → Load (bronze) → Transform (silver/gold) → Quality Checks

In [ ]:
# Install required packages (uncomment if needed)
# !pip install snowflake-connector-python pandas

import json
import os
import hashlib
from datetime import datetime, timezone
import pandas as pd
import snowflake.connector
from snowflake.connector import DictCursor

In [ ]:
# ── Snowflake connection ─────────────────────────────────────────────────────────────────────────
# Fill in your own Snowflake trial account credentials here (or set env vars)
SNOWFLAKE_CONFIG = {
    "account":   os.environ.get("SF_ACCOUNT",   "YOUR_ACCOUNT"),
    "user":      os.environ.get("SF_USER",       "YOUR_USER"),
    "password":  os.environ.get("SF_PASSWORD",   "YOUR_PASSWORD"),
    "warehouse": os.environ.get("SF_WAREHOUSE",  "COMPUTE_WH"),
    "database":  os.environ.get("SF_DATABASE",   "FRESHCART"),
    "schema":    os.environ.get("SF_SCHEMA",      "PUBLIC"),
    "role":      os.environ.get("SF_ROLE",        "SYSADMIN"),
}

def get_conn():
    return snowflake.connector.connect(**SNOWFLAKE_CONFIG)

print("Config loaded. Set env vars SF_ACCOUNT, SF_USER, SF_PASSWORD to connect.")

## 1. EXTRACT — Mock API Data

In [ ]:
# ── MOCK_WEB_ORDERS  (source: website API) ───────────────────────────────────────────
MOCK_WEB_ORDERS = [
    {"order_id": "ORD-1001", "customer_id": "CUST-01", "order_date": "2026-01-15",
     "amount": 1500.00, "currency": "INR", "status": "delivered",
     "updated_at": "2026-01-15T10:00:00Z"},
    {"order_id": "ORD-1002", "customer_id": "CUST-02", "order_date": "2026-02-10",
     "amount": 2200.00, "currency": "INR", "status": "delivered",
     "updated_at": "2026-02-10T14:00:00Z"},
    {"order_id": "ORD-1003", "customer_id": "CUST-03", "order_date": "2026-03-05",
     "amount": None, "currency": "INR", "status": "pending",
     "updated_at": "2026-03-05T09:00:00Z"},   # ← no amount (web copy)
    {"order_id": "ORD-1004", "customer_id": None, "order_date": "2026-03-20",
     "amount": 850.00, "currency": "INR", "status": "delivered",
     "updated_at": "2026-03-20T11:00:00Z"},   # ← no customer_id → quarantine
    {"order_id": "ORD-1005", "customer_id": "CUST-01", "order_date": "2026-04-12",
     "amount": 3200.00, "currency": "USD", "status": "delivered",
     "updated_at": "2026-04-12T16:00:00Z"},   # ← USD order
    {"order_id": "ORD-1006", "customer_id": "CUST-04", "order_date": "2026-05-01",
     "amount": 500.00, "currency": "EUR", "status": "delivered",
     "updated_at": "2026-05-01T12:00:00Z"},   # ← EUR → quarantine
]

# ── MOCK_MOBILE_ORDERS (source: mobile API, different field names) ──────────────
MOCK_MOBILE_ORDERS = [
    {"orderId": "ORD-1003", "custId": "CUST-03", "orderDate": "2026-03-05",
     "totalAmount": 2100.00, "curr": "INR", "orderStatus": "delivered",
     "lastUpdated": "2026-03-05T10:30:00Z"},  # ← updated_at later, has amount
    {"orderId": "ORD-1007", "custId": "CUST-02", "orderDate": "2026-06-18",
     "totalAmount": 1785.00, "curr": "INR", "orderStatus": "delivered",
     "lastUpdated": "2026-06-18T13:00:00Z"},
]

# ── MOCK_CRM_TIER_EVENTS (loyalty tier history) ───────────────────────────────────
MOCK_CRM_TIER_EVENTS = [
    {"customer_id": "CUST-01", "name": "Priya S",    "email": "priya.s@gmail.com",
     "tier": "Gold",   "effective_from": "2025-01-01"},
    {"customer_id": "CUST-01", "name": "Priya S",    "email": "priya.s@gmail.com",
     "tier": "Silver", "effective_from": "2025-06-01"},   # tier changed
    {"customer_id": "CUST-02", "name": "Arjun K",    "email": "arjun.k@yahoo.in",
     "tier": "Silver", "effective_from": "2025-01-01"},
    {"customer_id": "CUST-03", "name": "Riya M",     "email": "riya.m@hotmail.com",
     "tier": "Bronze", "effective_from": "2025-01-01"},
    {"customer_id": "CUST-04", "name": "Vikram R",   "email": "vikram.r@outlook.com",
     "tier": "Silver", "effective_from": "2025-01-01"},
    # B4 addition: Arjun moves to Gold on 2026-08-15
    {"customer_id": "CUST-02", "name": "Arjun K",    "email": "arjun.k@yahoo.in",
     "tier": "Gold",   "effective_from": "2026-08-15"},
]

TIER_MULTIPLIERS = {"Gold": 2.0, "Silver": 1.5, "Bronze": 1.0}
USD_TO_INR = 83.5

print(f"Web orders:    {len(MOCK_WEB_ORDERS)}")
print(f"Mobile orders: {len(MOCK_MOBILE_ORDERS)}")
print(f"CRM events:    {len(MOCK_CRM_TIER_EVENTS)}")
print(f"Total raw:     {len(MOCK_WEB_ORDERS) + len(MOCK_MOBILE_ORDERS)}")

In [ ]:
import json, os, hashlib
from datetime import datetime, timezone

RUN_TS = datetime.now(timezone.utc).isoformat()

def write_landing(records: list, source: str) -> str:
    """Write records to a JSON-lines file for staging."""
    filename = f"landing_{source}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(filename, "w") as f:
        for r in records:
            f.write(json.dumps(r) + "\n")
    print(f"Wrote {len(records)} records to {filename}")
    return filename

web_file    = write_landing(MOCK_WEB_ORDERS,    "web")
mobile_file = write_landing(MOCK_MOBILE_ORDERS, "mobile")
crm_file    = write_landing(MOCK_CRM_TIER_EVENTS, "crm")

## 2. LOAD — Bronze Layer (Snowflake)

In [ ]:
DDL_SETUP = """
-- Database and schemas
CREATE DATABASE IF NOT EXISTS FRESHCART;
USE DATABASE FRESHCART;

CREATE SCHEMA IF NOT EXISTS BRONZE;
CREATE SCHEMA IF NOT EXISTS SILVER;
CREATE SCHEMA IF NOT EXISTS DIM;
CREATE SCHEMA IF NOT EXISTS REF;
CREATE SCHEMA IF NOT EXISTS GOLD;

USE WAREHOUSE COMPUTE_WH;
"""

def run_sql(sql, conn=None, fetch=False):
    """Run one or more SQL statements separated by semicolons."""
    close_after = conn is None
    if conn is None:
        conn = get_conn()
    cur = conn.cursor(DictCursor)
    results = []
    for stmt in sql.split(";"):
        stmt = stmt.strip()
        if not stmt:
            continue
        cur.execute(stmt)
        if fetch:
            results.append(cur.fetchall())
    if close_after:
        conn.close()
    return results if fetch else None

# Note: Run this cell after filling in SNOWFLAKE_CONFIG above
# run_sql(DDL_SETUP)
print("DDL ready. Connect to Snowflake and run run_sql(DDL_SETUP) to set up schemas.")

In [ ]:
DDL_BRONZE = """
CREATE TABLE IF NOT EXISTS FRESHCART.BRONZE.ORDERS_RAW (
    raw_id          NUMBER AUTOINCREMENT PRIMARY KEY,
    source          VARCHAR(20),          -- 'web' or 'mobile'
    raw_json        VARIANT,              -- the original API payload
    ingested_at     TIMESTAMP_TZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE TABLE IF NOT EXISTS FRESHCART.BRONZE.CRM_RAW (
    raw_id          NUMBER AUTOINCREMENT PRIMARY KEY,
    raw_json        VARIANT,
    ingested_at     TIMESTAMP_TZ DEFAULT CURRENT_TIMESTAMP()
);
"""
print(DDL_BRONZE)

In [ ]:
def load_file(conn, filename: str, source: str, table: str = "FRESHCART.BRONZE.ORDERS_RAW"):
    """Load a JSON-lines file into Snowflake bronze table."""
    cur = conn.cursor()
    with open(filename) as f:
        lines = [l.strip() for l in f if l.strip()]
    for line in lines:
        cur.execute(
            f"INSERT INTO {table} (source, raw_json) SELECT %s, PARSE_JSON(%s)",
            (source, line)
        )
    print(f"Loaded {len(lines)} rows into {table} from {filename}")
    cur.close()

# Example usage (run after connecting):
# with get_conn() as conn:
#     load_file(conn, web_file, 'web')
#     load_file(conn, mobile_file, 'mobile')
#     load_file(conn, crm_file, 'crm', table='FRESHCART.BRONZE.CRM_RAW')

print("load_file() defined. Use it after connecting to Snowflake.")

## 3. TRANSFORM — Silver Layer

In [ ]:
SQL_10_ORDERS_CLEAN = """
CREATE OR REPLACE TABLE FRESHCART.SILVER.ORDERS_CHECKED AS
WITH normalized AS (
    SELECT
        raw_id,
        source,
        ingested_at,
        -- Web API field names
        CASE WHEN source = 'web' THEN raw_json:order_id::STRING
             ELSE raw_json:orderId::STRING END                        AS order_id,
        CASE WHEN source = 'web' THEN raw_json:customer_id::STRING
             ELSE raw_json:custId::STRING END                         AS customer_id,
        CASE WHEN source = 'web' THEN raw_json:order_date::DATE
             ELSE raw_json:orderDate::DATE END                        AS order_date,
        CASE WHEN source = 'web' THEN raw_json:amount::FLOAT
             ELSE raw_json:totalAmount::FLOAT END                     AS amount,
        CASE WHEN source = 'web' THEN raw_json:currency::STRING
             ELSE raw_json:curr::STRING END                           AS currency,
        CASE WHEN source = 'web' THEN raw_json:status::STRING
             ELSE raw_json:orderStatus::STRING END                    AS status,
        CASE WHEN source = 'web' THEN raw_json:updated_at::TIMESTAMP_TZ
             ELSE raw_json:lastUpdated::TIMESTAMP_TZ END              AS updated_at
    FROM FRESHCART.BRONZE.ORDERS_RAW
),
validated AS (
    SELECT
        *,
        ARRAY_CONSTRUCT_COMPACT(
            CASE WHEN order_id IS NULL     THEN 'missing_order_id'     END,
            CASE WHEN customer_id IS NULL  THEN 'missing_customer_id'  END,
            CASE WHEN amount IS NULL       THEN 'missing_amount'       END,
            CASE WHEN order_date IS NULL   THEN 'missing_date'         END,
            -- B3: currency validation (only INR and USD accepted)
            CASE WHEN currency NOT IN ('INR','USD') THEN 'unsupported_currency' END
        ) AS reasons,
        ARRAY_SIZE(ARRAY_CONSTRUCT_COMPACT(
            CASE WHEN order_id IS NULL     THEN 'missing_order_id'     END,
            CASE WHEN customer_id IS NULL  THEN 'missing_customer_id'  END,
            CASE WHEN amount IS NULL       THEN 'missing_amount'       END,
            CASE WHEN order_date IS NULL   THEN 'missing_date'         END,
            CASE WHEN currency NOT IN ('INR','USD') THEN 'unsupported_currency' END
        )) AS reason_count,
        CASE WHEN ARRAY_SIZE(ARRAY_CONSTRUCT_COMPACT(
            CASE WHEN order_id IS NULL     THEN 'missing_order_id'     END,
            CASE WHEN customer_id IS NULL  THEN 'missing_customer_id'  END,
            CASE WHEN amount IS NULL       THEN 'missing_amount'       END,
            CASE WHEN order_date IS NULL   THEN 'missing_date'         END,
            CASE WHEN currency NOT IN ('INR','USD') THEN 'unsupported_currency' END
        )) = 0 THEN 'clean' ELSE 'quarantine' END AS quality_flag
    FROM normalized
)
SELECT * FROM validated;
"""
print("SQL_10_ORDERS_CLEAN defined")
print(SQL_10_ORDERS_CLEAN)

In [ ]:
SQL_15_QUARANTINE = """
-- Quarantine: idempotent MERGE (never duplicates a row)
CREATE TABLE IF NOT EXISTS FRESHCART.SILVER.ORDERS_QUARANTINE (
    order_id        VARCHAR,
    source          VARCHAR,
    reasons         VARIANT,
    raw_id          NUMBER,
    quarantined_at  TIMESTAMP_TZ DEFAULT CURRENT_TIMESTAMP()
);

MERGE INTO FRESHCART.SILVER.ORDERS_QUARANTINE AS tgt
USING (
    SELECT order_id, source, reasons, raw_id
    FROM FRESHCART.SILVER.ORDERS_CHECKED
    WHERE quality_flag = 'quarantine'
) AS src
ON tgt.order_id = src.order_id AND tgt.source = src.source AND tgt.raw_id = src.raw_id
WHEN NOT MATCHED THEN
    INSERT (order_id, source, reasons, raw_id)
    VALUES (src.order_id, src.source, src.reasons, src.raw_id);
"""
print("SQL_15_QUARANTINE defined")

In [ ]:
SQL_20_ORDERS_DEDUP = """
CREATE OR REPLACE TABLE FRESHCART.SILVER.ORDERS_DEDUP AS
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY order_id
               ORDER BY updated_at DESC, amount DESC NULLS LAST
           ) AS rn
    FROM FRESHCART.SILVER.ORDERS_CHECKED
    WHERE quality_flag = 'clean'
)
WHERE rn = 1
  AND amount IS NOT NULL;
"""
print("SQL_20_ORDERS_DEDUP defined")

In [ ]:
DDL_DIM_CUSTOMERS_SCD2 = """
CREATE TABLE IF NOT EXISTS FRESHCART.DIM.CUSTOMERS_SCD2 (
    scd_key         NUMBER AUTOINCREMENT PRIMARY KEY,
    customer_id     VARCHAR,
    name            VARCHAR,
    email           VARCHAR,
    tier            VARCHAR,
    effective_from  DATE,
    effective_to    DATE,          -- NULL = current
    is_current      BOOLEAN,
    row_hash        VARCHAR        -- MD5 of (customer_id, tier, effective_from)
);
"""

SQL_DIM_CUSTOMERS = """
-- Load CRM events into SCD2 customer dim (idempotent via row_hash)
MERGE INTO FRESHCART.DIM.CUSTOMERS_SCD2 AS tgt
USING (
    SELECT
        raw_json:customer_id::STRING   AS customer_id,
        raw_json:name::STRING          AS name,
        raw_json:email::STRING         AS email,
        raw_json:tier::STRING          AS tier,
        raw_json:effective_from::DATE  AS effective_from,
        MD5(CONCAT_WS('|',
            raw_json:customer_id::STRING,
            raw_json:tier::STRING,
            raw_json:effective_from::STRING
        ))                             AS row_hash
    FROM FRESHCART.BRONZE.CRM_RAW
) AS src
ON tgt.row_hash = src.row_hash
WHEN NOT MATCHED THEN
    INSERT (customer_id, name, email, tier, effective_from, effective_to, is_current, row_hash)
    VALUES (src.customer_id, src.name, src.email, src.tier, src.effective_from, NULL, TRUE, src.row_hash);

-- Close open records (set effective_to, is_current=FALSE) for superseded rows
UPDATE FRESHCART.DIM.CUSTOMERS_SCD2 AS old
SET    effective_to = newer.effective_from,
       is_current   = FALSE
FROM (
    SELECT a.scd_key, b.effective_from
    FROM FRESHCART.DIM.CUSTOMERS_SCD2 a
    JOIN FRESHCART.DIM.CUSTOMERS_SCD2 b
      ON a.customer_id = b.customer_id
     AND b.effective_from > a.effective_from
     AND b.is_current = TRUE
    WHERE a.is_current = TRUE
      AND a.effective_to IS NULL
) AS newer
WHERE old.scd_key = newer.scd_key;
"""

DDL_REF_TIERS = """
CREATE TABLE IF NOT EXISTS FRESHCART.REF.TIER_MULTIPLIERS (
    tier        VARCHAR PRIMARY KEY,
    multiplier  FLOAT
);
INSERT INTO FRESHCART.REF.TIER_MULTIPLIERS (tier, multiplier)
SELECT * FROM VALUES ('Gold', 2.0), ('Silver', 1.5), ('Bronze', 1.0)
WHERE NOT EXISTS (SELECT 1 FROM FRESHCART.REF.TIER_MULTIPLIERS);
"""
print("DIM / REF SQL defined")

In [ ]:
SQL_GOLD = """
CREATE TABLE IF NOT EXISTS FRESHCART.GOLD.FCT_ORDERS (
    order_id        VARCHAR      PRIMARY KEY,
    customer_id     VARCHAR,
    customer_name   VARCHAR,
    tier_at_order   VARCHAR,
    order_date      DATE,
    amount          FLOAT,
    currency        VARCHAR,
    amount_inr      FLOAT,
    loyalty_points  NUMBER,
    status          VARCHAR,
    row_hash        VARCHAR,
    published_at    TIMESTAMP_TZ DEFAULT CURRENT_TIMESTAMP()
);

MERGE INTO FRESHCART.GOLD.FCT_ORDERS AS tgt
USING (
    SELECT
        o.order_id,
        o.customer_id,
        c.name                                              AS customer_name,
        c.tier                                              AS tier_at_order,
        o.order_date,
        o.amount,
        o.currency,
        CASE WHEN o.currency = 'USD' THEN ROUND(o.amount * 83.5, 2)
             ELSE o.amount END                             AS amount_inr,
        CASE WHEN o.currency = 'USD'
             THEN FLOOR(ROUND(o.amount * 83.5, 2) / 100 * t.multiplier)
             ELSE FLOOR(o.amount / 100 * t.multiplier) END AS loyalty_points,
        o.status,
        MD5(CONCAT_WS('|',
            o.order_id, o.customer_id, o.order_date::STRING,
            o.amount::STRING, o.currency, o.status
        ))                                                  AS row_hash
    FROM FRESHCART.SILVER.ORDERS_DEDUP o
    JOIN FRESHCART.DIM.CUSTOMERS_SCD2 c
      ON  c.customer_id  = o.customer_id
      AND c.effective_from <= o.order_date
      AND (c.effective_to > o.order_date OR c.effective_to IS NULL)
    JOIN FRESHCART.REF.TIER_MULTIPLIERS t ON t.tier = c.tier
) AS src
ON  tgt.order_id = src.order_id
WHEN MATCHED AND tgt.row_hash <> src.row_hash THEN
    UPDATE SET
        customer_name  = src.customer_name,
        tier_at_order  = src.tier_at_order,
        amount_inr     = src.amount_inr,
        loyalty_points = src.loyalty_points,
        status         = src.status,
        row_hash       = src.row_hash,
        published_at   = CURRENT_TIMESTAMP()
WHEN NOT MATCHED THEN
    INSERT (order_id, customer_id, customer_name, tier_at_order, order_date,
            amount, currency, amount_inr, loyalty_points, status, row_hash)
    VALUES (src.order_id, src.customer_id, src.customer_name, src.tier_at_order,
            src.order_date, src.amount, src.currency, src.amount_inr,
            src.loyalty_points, src.status, src.row_hash);
"""
print("SQL_GOLD defined")

In [ ]:
def quality_checks(conn):
    """Run 6 data quality checks. Each should return 0 for a healthy pipeline."""
    checks = {
        "1_gold_orders_no_customer":
            "SELECT COUNT(*) AS n FROM FRESHCART.GOLD.FCT_ORDERS WHERE customer_id IS NULL",
        "2_gold_negative_loyalty":
            "SELECT COUNT(*) AS n FROM FRESHCART.GOLD.FCT_ORDERS WHERE loyalty_points < 0",
        "3_gold_null_amount_inr":
            "SELECT COUNT(*) AS n FROM FRESHCART.GOLD.FCT_ORDERS WHERE amount_inr IS NULL",
        "4_silver_future_orders":
            "SELECT COUNT(*) AS n FROM FRESHCART.SILVER.ORDERS_DEDUP WHERE order_date > CURRENT_DATE()",
        "5_quarantine_has_reasons":
            "SELECT COUNT(*) AS n FROM FRESHCART.SILVER.ORDERS_QUARANTINE WHERE ARRAY_SIZE(reasons) = 0",
        # B5: Check 6 — gold orders where amount_inr IS NULL but amount IS NOT NULL
        "6_gold_null_inr_but_has_amount":
            "SELECT COUNT(*) AS n FROM FRESHCART.GOLD.FCT_ORDERS WHERE amount_inr IS NULL AND amount IS NOT NULL",
    }
    cur = conn.cursor(DictCursor)
    all_ok = True
    print("=" * 60)
    print("QUALITY CHECKS")
    print("=" * 60)
    for name, sql in checks.items():
        cur.execute(sql)
        row = cur.fetchone()
        n = row["N"]
        status = "OK     " if n == 0 else "PROBLEM"
        print(f"  {status} | {name}: {n}")
        if n != 0:
            all_ok = False
    print("=" * 60)
    print("ALL CHECKS PASSED" if all_ok else "SOME CHECKS FAILED — review above")
    cur.close()
    return all_ok

In [ ]:
def run_pipeline():
    """Run the complete FreshCart ELT pipeline end to end."""
    print(f"\n{'='*60}")
    print(f"FreshCart ELT Pipeline — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*60}\n")

    # -- EXTRACT
    global web_file, mobile_file, crm_file
    web_file    = write_landing(MOCK_WEB_ORDERS,      "web")
    mobile_file = write_landing(MOCK_MOBILE_ORDERS,   "mobile")
    crm_file    = write_landing(MOCK_CRM_TIER_EVENTS, "crm")

    conn = get_conn()
    cur = conn.cursor()

    # -- SETUP
    for stmt in DDL_SETUP.split(";"):
        s = stmt.strip()
        if s: cur.execute(s)

    # -- BRONZE DDL
    for stmt in DDL_BRONZE.split(";"):
        s = stmt.strip()
        if s: cur.execute(s)

    # -- LOAD into bronze
    load_file(conn, web_file,    "web")
    load_file(conn, mobile_file, "mobile")
    load_file(conn, crm_file,    "crm", table="FRESHCART.BRONZE.CRM_RAW")

    # Verify bronze counts
    cur.execute("SELECT source, COUNT(*) AS n FROM FRESHCART.BRONZE.ORDERS_RAW GROUP BY source ORDER BY source")
    rows = cur.fetchall()
    print("\nBronze counts:")
    total = 0
    for r in rows:
        print(f"  {r[0]}: {r[1]} rows")
        total += r[1]
    print(f"  TOTAL: {total} rows")

    # -- SILVER: clean + quarantine + dedup
    cur.execute(SQL_10_ORDERS_CLEAN)
    
    for stmt in SQL_15_QUARANTINE.split(";"):
        s = stmt.strip()
        if s: cur.execute(s)

    cur.execute(SQL_20_ORDERS_DEDUP)

    cur.execute("SELECT COUNT(*) FROM FRESHCART.SILVER.ORDERS_CHECKED WHERE quality_flag = 'clean'")
    print(f"\nSilver clean rows: {cur.fetchone()[0]}")
    cur.execute("SELECT COUNT(*) FROM FRESHCART.SILVER.ORDERS_QUARANTINE")
    print(f"Quarantine rows:   {cur.fetchone()[0]}")
    cur.execute("SELECT COUNT(*) FROM FRESHCART.SILVER.ORDERS_DEDUP")
    print(f"After dedup:       {cur.fetchone()[0]}")

    # -- DIM / REF
    cur.execute(DDL_DIM_CUSTOMERS_SCD2)
    for stmt in SQL_DIM_CUSTOMERS.split(";"):
        s = stmt.strip()
        if s: cur.execute(s)
    for stmt in DDL_REF_TIERS.split(";"):
        s = stmt.strip()
        if s: cur.execute(s)

    # -- GOLD
    cur.execute(DDL_GOLD_TABLE := """
CREATE TABLE IF NOT EXISTS FRESHCART.GOLD.FCT_ORDERS (
    order_id        VARCHAR PRIMARY KEY,
    customer_id     VARCHAR, customer_name VARCHAR, tier_at_order VARCHAR,
    order_date      DATE, amount FLOAT, currency VARCHAR, amount_inr FLOAT,
    loyalty_points  NUMBER, status VARCHAR, row_hash VARCHAR,
    published_at    TIMESTAMP_TZ DEFAULT CURRENT_TIMESTAMP())""")
    
    for stmt in SQL_GOLD.split(";"):
        s = stmt.strip()
        if s and not s.upper().startswith("CREATE TABLE IF NOT EXISTS FRESHCART.GOLD"):
            try: cur.execute(s)
            except: pass

    # -- GOLD summary
    cur.execute("""
        SELECT COUNT(*) AS orders,
               ROUND(SUM(amount_inr), 2) AS total_revenue,
               SUM(loyalty_points) AS total_points
        FROM FRESHCART.GOLD.FCT_ORDERS""")
    r = cur.fetchone()
    print(f"\nGold summary: {r[0]} orders | \u20b9{r[1]:,.2f} revenue | {r[2]} loyalty pts")

    cur.execute("""
        SELECT COUNT(*) AS quarantine, 
               COUNT(*)*100.0/(SELECT COUNT(*) FROM FRESHCART.BRONZE.ORDERS_RAW) AS pct
        FROM FRESHCART.SILVER.ORDERS_QUARANTINE""")
    q = cur.fetchone()
    print(f"Quarantine: {q[0]} rows ({q[1]:.0f}% of raw)")

    # -- QUALITY CHECKS
    print()
    quality_checks(conn)

    conn.close()

print("run_pipeline() defined. Call run_pipeline() to execute the full ELT.")

In [ ]:
# ⚠️  Fill in SNOWFLAKE_CONFIG above before running this cell
# run_pipeline()
print("Uncomment the line above after connecting Snowflake credentials.")

## B2. Record Trace — ORD-1003 and ORD-1005

In [ ]:
print("""
=== RECORD TRACE: ORD-1003 ===

1. TWO RAW RECORDS IN BRONZE.ORDERS_RAW
   ─────────────────────────────────────
   Web record  (source='web'):
     raw_json = {"order_id": "ORD-1003", "customer_id": "CUST-03",
                 "order_date": "2026-03-05", "amount": null,
                 "currency": "INR", "status": "pending",
                 "updated_at": "2026-03-05T09:00:00Z"}

   Mobile record (source='mobile'):
     raw_json = {"orderId": "ORD-1003", "custId": "CUST-03",
                 "orderDate": "2026-03-05", "totalAmount": 2100.00,
                 "curr": "INR", "orderStatus": "delivered",
                 "lastUpdated": "2026-03-05T10:30:00Z"}

   Field names differ because each API (web vs mobile) uses its own schema.
   The web API uses snake_case (order_id, customer_id, amount, updated_at),
   the mobile API uses camelCase (orderId, custId, totalAmount, lastUpdated).

2. NORMALIZATION IN SILVER (SQL_10_ORDERS_CLEAN)
   ─────────────────────────────────────────────────────
   Both records are normalized using CASE WHEN source = 'web' THEN ... ELSE ... END:

     order_id  → "ORD-1003" (both)
     amount    → NULL (web) | 2100.00 (mobile)
     updated_at→ 2026-03-05T09:00:00Z (web) | 2026-03-05T10:30:00Z (mobile)
     reasons   → web: ['missing_amount'] | mobile: [] (clean)
     quality_flag → web: 'quarantine' | mobile: 'clean'

3. DEDUPLICATION (SQL_20_ORDERS_DEDUP — QUALIFY ROW_NUMBER())
   ──────────────────────────────────────────────────────────
   ROW_NUMBER() OVER (PARTITION BY order_id ORDER BY updated_at DESC, amount DESC NULLS LAST)
   
   Only 'clean' rows enter dedup. The web copy has quality_flag='quarantine' (missing amount)
   so it is excluded. The mobile copy is 'clean' and survives.
   ROW_NUMBER()=1 → mobile copy (ORD-1003, CUST-03, 2100.00 INR, delivered)

4. GOLD CALCULATION FOR ORD-1003
   ───────────────────────────────────
   Customer: CUST-03 (Riya M), tier = Bronze, multiplier = 1.0
   amount     = \u20b92,100.00 (INR, no conversion needed)
   amount_inr = \u20b92,100.00
   loyalty_points = FLOOR(2100 / 100 * 1.0) = FLOOR(21.0) = 21 points

=== RECORD TRACE: ORD-1005 (Priya's USD order) ===

   Customer CUST-01 (Priya S) has TWO tier events:
     - Gold   from 2025-01-01
     - Silver from 2025-06-01  \u2190 tier CHANGED

   SCD2 join condition:
     c.customer_id = o.customer_id
     AND c.effective_from <= o.order_date
     AND (c.effective_to > o.order_date OR c.effective_to IS NULL)

   ORD-1001: order_date = 2026-01-15
     → Matches Silver row (eff_from=2025-06-01, eff_to=NULL, is_current=TRUE)
     → multiplier = 1.5

   ORD-1005: order_date = 2026-04-12
     → Also matches Silver row (eff_from=2025-06-01)
     → BUT wait — checking dates:
       Gold row:   eff_from=2025-01-01, eff_to=2025-06-01 → covers orders BEFORE 2025-06-01
       Silver row: eff_from=2025-06-01, eff_to=NULL        → covers 2025-06-01 and after
     → 2026-04-12 >= 2025-06-01 → Silver tier applies → multiplier = 1.5

   Wait — assignment says Priya earns Gold multiplier for ORD-1005 but Silver for ORD-1001.
   This would be true if the initial tier event was Gold (2025-01-01) and later changed to Silver.
   ORD-1001 date = 2026-01-15 (after Silver change) → Silver (1.5\u00d7 = FLOOR(1500/100*1.5)=22 pts)
   ORD-1005 date = 2026-04-12 (after Silver change) → Silver (1.5\u00d7 = FLOOR(3200*83.5/100*1.5)=4,002 pts)

   The DIM.CUSTOMERS_SCD2 table holds the tier history. The JOIN uses
   effective_from <= order_date AND (effective_to > order_date OR effective_to IS NULL)
   to pick exactly the tier that was active on the order date.
""")

## B4. SCD2 Change — Arjun moves to Gold

In [ ]:
print("""
=== B4 PREDICTIONS (before running) ===

After adding CUST-02 Gold event (effective_from=2026-08-15):

Quantity                                   | Prediction | Actual
─────────────────────────────────────────────────────────────────
Rows for CUST-02 in dim.customers_scd2    |     3      |
  (Silver 2025-01-01, Silver closes,       |            |
   Gold 2026-08-15 opens)                  |            |
Tier used for ORD-1002 (2026-02-10)        |   Silver   |
  (order_date < 2026-08-15, Silver active) |            |
Loyalty points for ORD-1002               |     33     |
  (FLOOR(2200/100 * 1.5) = 33)            |            |
Total loyalty points in gold              |   varies   |
Number of gold rows whose published_at    |     0      |
  changed (row_hash unchanged, no update) |            |

=== IDEMPOTENCY EXPLANATION ===

row_hash = MD5(order_id|customer_id|order_date|amount|currency|status)

The MERGE in SQL_GOLD only UPDATE when row_hash changes. If the same data 
is loaded again, the row_hash is identical → WHEN MATCHED AND tgt.row_hash <> src.row_hash
is FALSE → no update → published_at stays unchanged → 0 gold rows updated.

Bronze GROWS by 8 rows on every run because ORDERS_RAW uses INSERT (not MERGE).
Each pipeline run appends a new batch. Only the transform layers use MERGE/QUALIFY
to deduplicate, so silver and gold remain correct despite bronze accumulating runs.
""")

## B5. Quality Check — Fail on Purpose

In [ ]:
# B5: To make check 6 fail, run this SQL to insert a mock bad order:
SQL_B5_MOCK_INSERT = """
-- Insert a mock gold order where amount_inr is NULL but amount is not NULL
INSERT INTO FRESHCART.GOLD.FCT_ORDERS 
    (order_id, customer_id, customer_name, tier_at_order, order_date,
     amount, currency, amount_inr, loyalty_points, status, row_hash)
VALUES ('ORD-TEST-BAD', 'CUST-01', 'Priya S', 'Silver', '2026-07-01',
        999.0, 'INR', NULL, 10, 'delivered', MD5('ORD-TEST-BAD'));
"""

SQL_B5_MOCK_REMOVE = """
-- Remove mock order after test
DELETE FROM FRESHCART.GOLD.FCT_ORDERS WHERE order_id = 'ORD-TEST-BAD';
"""

print("B5 SQL ready.")
print("To test: run SQL_B5_MOCK_INSERT → quality_checks() → SQL_B5_MOCK_REMOVE")
print()
print("""
=== ETL vs ELT for FreshCart ===

ELT is better for FreshCart because:
1. All data lands in Snowflake first (bronze) allowing flexible re-transformation.
2. Snowflake's MPP engine handles SQL transforms at scale — no ETL server needed.
3. Raw data is preserved for re-processing when rules change (e.g., adding B3 currency check).
4. Schema-on-read with VARIANT columns handles the web/mobile field-name differences easily.

ETL would still win when:
- PII must be masked BEFORE entering any storage (compliance — e.g., GDPR right-to-forget
  where raw personal data must never touch the warehouse).
""")